In [1]:
%load_ext autoreload
%autoreload 2

# NeoOLAF XQuality Machine 32 layer-wise ablation notebook

This notebook is intended to live at:

```text
NeoOLAF/examples/XQualityMachine32/layer_ablation_machine32.ipynb
```

It assumes `NeoOLAF` is the repository root and that the package source is under `src/`.

Main uses:

1. Run full or partial NeoOLAF pipelines on the Machine 32 PDF.
2. Resume from a saved layer `state.json`.
3. Review prompt size and prompt content layer by layer.
4. Evaluate each saved layer state against XQuality Machine 32 ground truth in JSON or Excel format.


In [2]:
from pathlib import Path
import sys
import json
import pandas as pd

# Notebook location: NeoOLAF/examples/XQualityMachine32/
ROOT = Path.cwd()
if ROOT.name == "XQualityMachine32":
    ROOT = ROOT.parents[1]
elif (ROOT / "src" / "neoolaf").exists():
    ROOT = ROOT
else:
    # Useful when running from another working directory.
    ROOT = Path("../..").resolve()

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("NeoOLAF root:", ROOT)
print("src exists:", SRC.exists())

NeoOLAF root: c:\Users\henri\Documents\git\post-doc\NeoOLAF
src exists: True


## 1. Configure paths

Update these paths to match your local repository. The notebook accepts either a JSON ground-truth file or an Excel file. If an Excel file is used, it is converted automatically to JSON next to the original file.


In [7]:
# Input PDF and gold truth
PDF_PATH = ROOT / "data" / "XQuality" / "Textual" / "Chapitre_8_Alarmes_et_messages.pdf"
GOLD_JSON = ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"
GOLD_EXCEL = ROOT / "data" / "XQuality" / "Machine32" / "machine32_triples.xlsx"

# Seed ontology, optional for the current minimal CLI.
SEED_ONTOLOGY = ROOT / "data" / "ontology" / "ContextOntology-COInd4.owl"

# Run folders
RUN_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "full"
EVAL_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32" / "eval_layers"

# Model and profile
MODEL = "openrouter/openai/gpt-oss-20b"
PROFILE = "xquality_loose"  # alternatives: xquality_strict_extraction, xquality_loose, xquality_relaxed_recall

print("PDF_PATH:", PDF_PATH)
print("GOLD_JSON:", GOLD_JSON)
print("GOLD_EXCEL:", GOLD_EXCEL)
print("RUN_DIR:", RUN_DIR)

PDF_PATH: c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Textual\Chapitre_8_Alarmes_et_messages.pdf
GOLD_JSON: c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Examples\XQuality_all_triplets_flat_en.json
GOLD_EXCEL: c:\Users\henri\Documents\git\post-doc\NeoOLAF\data\XQuality\Machine32\machine32_triples.xlsx
RUN_DIR: c:\Users\henri\Documents\git\post-doc\NeoOLAF\examples\XQualityMachine32\runs\xquality_machine32\full


## 2. Optional: convert Excel ground truth to JSON

The evaluation helper accepts both `.json` and `.xlsx`, but converting once makes the experiment more reproducible.


In [4]:
from neoolaf.evaluation.runners.evaluate_layer_state import convert_xquality_excel_to_json

if GOLD_EXCEL.exists() and not GOLD_JSON.exists():
    convert_xquality_excel_to_json(GOLD_EXCEL, GOLD_JSON)
    print("Converted Excel ground truth to:", GOLD_JSON)
else:
    print("No conversion needed.")

No conversion needed.


## 3. Run the pipeline

The commands below use the CLI added for ablation. They save a restartable `state.json` after each executed layer.

Full run:


In [5]:
# Full run, layers 0 to 12.
# Uncomment when ready to execute.

# !python -m neoolaf run \
#   --input-pdf "{PDF_PATH}" \
#   --run-dir "{RUN_DIR}" \
#   --from-layer 0 \
#   --to-layer 12 \
#   --model "{MODEL}" \
#   --rag-backend agentic \
#   --verbose

### Partial examples

Layer 0 to 1 only:


In [10]:
LAYER01_RUN_DIR = ROOT / "runs" / "xquality_machine32" / "layer01_debug"

!python -m neoolaf run \
   --input-pdf "{PDF_PATH}" \
   --run-dir "{LAYER01_RUN_DIR}" \
   --from-layer 0 \
   --to-layer 1 \
   --model "{MODEL}" \
   --rag-backend agentic \
   --max-chunks-layer01 3 \
   --verbose

[NeoOLAF] Run directory: c:\Users\henri\Documents\git\post-doc\NeoOLAF\runs\xquality_machine32\layer01_debug
[NeoOLAF] from_layer=0, to_layer=1, skip_layers=[]
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 3.50s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction
[NeoOLAF][layer01_linguistic_expression_extraction] JSON parse failed for chunk chunk_0000; saved raw response and continued.
[NeoOLAF][layer01_linguistic_expression_extraction] JSON parse failed for chunk chunk_0001; saved raw response and continued.
[NeoOLAF] Finished layer: layer01_linguistic_expression_extraction in 73.74s
[NeoOLAF] Pipeline finished in 77.59s
[NeoOLAF] Saved checkpoint: c:\Users\henri\Documents\git\post-

01:06:40 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
01:06:40 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(

Layer 1 - chunks: 100%|██████████| 3/3 [01:13<00:00, 31.64s/it]
                                                               


01:06:24 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
01:06:24 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'
c:\Users\henri\Documents\git\post-doc\neoolafvenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(

Layer 1 - chunks: 100%|██████████| 55/55 [33:54<00:00, 15.04s/it]
                                                                 


[NeoOLAF] Run directory: c:\Users\henri\Documents\git\post-doc\NeoOLAF\runs\xquality_machine32\layer01_only
[NeoOLAF] from_layer=0, to_layer=1, skip_layers=[]
[NeoOLAF] Pipeline has 13 layers
[NeoOLAF] Selected layers: ['layer00_preprocessing', 'layer01_linguistic_expression_extraction']
[NeoOLAF] Layer 0/12: layer00_preprocessing

[NeoOLAF] Starting layer: layer00_preprocessing
[NeoOLAF] Finished layer: layer00_preprocessing in 3.40s
[NeoOLAF] Layer 1/12: layer01_linguistic_expression_extraction

[NeoOLAF] Starting layer: layer01_linguistic_expression_extraction
[NeoOLAF][layer01_linguistic_expression_extraction] JSON parse failed for chunk chunk_0001; saved raw response and continued.
[NeoOLAF][layer01_linguistic_expression_extraction] JSON parse failed for chunk chunk_0002; saved raw response and continued.
[NeoOLAF][layer01_linguistic_expression_extraction] JSON parse failed for chunk chunk_0003; saved raw response and continued.
[NeoOLAF][layer01_linguistic_expression_extraction] 

In [9]:
LAYER01_RUN_DIR = ROOT / "runs" / "xquality_machine32" / "layer01_only"

!python -m neoolaf run \
   --input-pdf "{PDF_PATH}" \
   --run-dir "{LAYER01_RUN_DIR}" \
   --from-layer 0 \
   --to-layer 1 \
   --model "{MODEL}" \
   --rag-backend agentic \
   --verbose

^C


Resume from layer 1 and run layers 2 and 3:


In [ ]:
LAYER02_03_RUN_DIR = ROOT / "runs" / "xquality_machine32" / "layer02_03_from_l1"
LAYER01_STATE = LAYER01_RUN_DIR / "layer01_linguistic_expression_extraction" / "state.json"

# !python -m neoolaf run \
#   --resume-from "{LAYER01_STATE}" \
#   --run-dir "{LAYER02_03_RUN_DIR}" \
#   --from-layer 2 \
#   --to-layer 3 \
#   --model "{MODEL}" \
#   --rag-backend agentic \
#   --verbose

Run an ablation by skipping layers 8 and 9:


In [ ]:
ABLATION_RUN_DIR = ROOT / "runs" / "xquality_machine32" / "skip_axiom_layers"
LAYER07_STATE = RUN_DIR / "layer07_hierarchisation" / "state.json"

# !python -m neoolaf run \
#   --resume-from "{LAYER07_STATE}" \
#   --run-dir "{ABLATION_RUN_DIR}" \
#   --from-layer 8 \
#   --to-layer 12 \
#   --skip-layers 8,9 \
#   --model "{MODEL}" \
#   --rag-backend agentic \
#   --verbose

## 4. Inspect prompt size layer by layer

Each LLM layer now saves:

```text
<run_dir>/<layer_name>/prompt_stats.json
<run_dir>/<layer_name>/prompts/prompt_001_system.txt
<run_dir>/<layer_name>/prompts/prompt_001_user.txt
<run_dir>/<layer_name>/prompts/prompt_001_full.txt
```


In [ ]:
def collect_prompt_stats(run_dir: Path) -> pd.DataFrame:
    rows = []
    for stats_path in sorted(run_dir.glob("layer*/prompt_stats.json")):
        data = json.loads(stats_path.read_text(encoding="utf-8"))
        rows.append({
            "layer": stats_path.parent.name,
            "prompt_call_count": data.get("prompt_call_count", 0),
            "max_estimated_tokens": data.get("max_estimated_tokens", 0),
            "average_estimated_tokens": data.get("average_estimated_tokens", 0),
            "total_estimated_tokens": data.get("total_estimated_tokens", 0),
            "max_prompt_chars": data.get("max_prompt_chars", 0),
        })
    return pd.DataFrame(rows)

prompt_df = collect_prompt_stats(RUN_DIR)
prompt_df

In [ ]:
# Show the largest prompts first.
if not prompt_df.empty:
    display(prompt_df.sort_values("max_estimated_tokens", ascending=False))
else:
    print("No prompt_stats.json files found yet. Run the pipeline first.")

## 5. Evaluate every saved layer state against Machine 32 ground truth

The early layers are evaluated mostly as semantic label coverage. From layer 4 onward, relation-level metrics become more meaningful. From layer 5 onward, triple-level precision/recall/F1 is the main signal.


In [ ]:
from neoolaf.evaluation.runners.evaluate_layer_state import evaluate_run_layers

GOLD_PATH = GOLD_JSON if GOLD_JSON.exists() else GOLD_EXCEL
print("Using gold path:", GOLD_PATH)

results = evaluate_run_layers(
    run_dir=RUN_DIR,
    gold_path=GOLD_PATH,
    profile_name=PROFILE,
    output_dir=EVAL_DIR,
)

print(f"Evaluated {len(results)} layer states.")

In [ ]:
def summarize_layer_results(results: list[dict]) -> pd.DataFrame:
    rows = []
    for item in results:
        extraction = item.get("extraction", {})
        entity = extraction.get("entity", {})
        relation = extraction.get("relation", {})
        counts = extraction.get("counts", {})
        prompt_stats = item.get("prompt_stats", {})
        rows.append({
            "layer": item.get("layer_name"),
            "entity_P": entity.get("precision"),
            "entity_R": entity.get("recall"),
            "entity_F1": entity.get("f1"),
            "relation_P": relation.get("precision"),
            "relation_R": relation.get("recall"),
            "relation_F1": relation.get("f1"),
            "pred_entities": counts.get("pred_entities"),
            "gold_entities": counts.get("gold_entities"),
            "pred_relations": counts.get("pred_relations"),
            "gold_relations": counts.get("gold_relations"),
            "prompt_calls": prompt_stats.get("prompt_call_count", 0),
            "max_prompt_tokens": prompt_stats.get("max_estimated_tokens", 0),
        })
    return pd.DataFrame(rows)

summary_df = summarize_layer_results(results)
summary_df

In [ ]:
# Save a compact CSV summary for reporting.
EVAL_DIR.mkdir(parents=True, exist_ok=True)
summary_csv = EVAL_DIR / "layer_ablation_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Saved:", summary_csv)

## 6. Plot layer-wise evolution

These plots help identify where recall, precision, or prompt size changes sharply.


In [ ]:
import matplotlib.pyplot as plt

if not summary_df.empty:
    ax = summary_df.plot(x="layer", y=["entity_F1", "relation_F1"], marker="o", figsize=(12, 5))
    ax.set_ylabel("F1")
    ax.set_title("Layer-wise entity and relation F1")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No evaluation summary available.")

In [ ]:
if not summary_df.empty:
    ax = summary_df.plot(x="layer", y="max_prompt_tokens", kind="bar", figsize=(12, 5), legend=False)
    ax.set_ylabel("Estimated tokens")
    ax.set_title("Maximum prompt size by layer")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No prompt statistics available.")

## 7. Inspect unmatched predictions for a specific layer

Use this to see what the layer is producing that does not match the ground truth.


In [ ]:
# Choose one layer result to inspect.
LAYER_TO_INSPECT = "layer05_candidate_triple_generation"

selected = next((item for item in results if item.get("layer_name") == LAYER_TO_INSPECT), None)
if selected is None:
    print("Layer result not found:", LAYER_TO_INSPECT)
else:
    unmatched = selected["extraction"].get("unmatched", {})
    print("Unmatched predicted relations, first 10:")
    for relation in unmatched.get("relations_pred", [])[:10]:
        print(relation)
    print("
Unmatched gold relations, first 10:")
    for relation in unmatched.get("relations_gold", [])[:10]:
        print(relation)